<a href="https://colab.research.google.com/github/vipinsinghbora03-dot/Python_Project_28/blob/main/project_28.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd


df = pd.read_csv("28_tournament_match_stats.csv")

print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

print("\nFirst 5 Rows:")
print(df.head())

Dataset Shape: (609, 10)

Columns:
Index(['Match_ID', 'Match_Date', 'Team', 'Opponent', 'Player_Name',
       'Player_Points', 'Match_Duration_Mins', 'Match_Type',
       'Tournament_Round', 'Result'],
      dtype='object')

Data Types:
Match_ID                object
Match_Date              object
Team                    object
Opponent                object
Player_Name             object
Player_Points          float64
Match_Duration_Mins    float64
Match_Type              object
Tournament_Round        object
Result                  object
dtype: object

First 5 Rows:
  Match_ID  Match_Date      Team Opponent Player_Name  Player_Points  \
0  MA00000  2026-05-26    Eagles  Falcons   Player 50           35.0   
1  MA00001  2026-05-20   Falcons   Eagles   Player 34           24.0   
2  MA00002  2025-06-25   Falcons    Lions   Player 65            NaN   
3  MA00003  2025-08-05    Sharks   Sharks   Player 10           31.0   
4  MA00004  2026-09-16  Panthers   Sharks   Player 10           

In [23]:
#  Check for missing values and duplicate Match_ID records

# Check missing values
print(df.isnull().sum())

# Check duplicate Match_ID
print(df["Match_ID"].duplicated().sum())

Match_ID                0
Match_Date              0
Team                    0
Opponent                0
Player_Name             0
Player_Points          15
Match_Duration_Mins    15
Match_Type              0
Tournament_Round        0
Result                  0
dtype: int64
9


In [24]:
# 1. Remove duplicate records based on Match_ID
if 'Match_ID' in df.columns:
    df = df.drop_duplicates(subset=['Match_ID'])

# 2. Handle missing values in critical columns using a robust method (median)
if 'Player_Points' in df.columns:
    df['Player_Points'] = df['Player_Points'].fillna(df['Player_Points'].median())

if 'Match_Duration_Mins' in df.columns:
    df['Match_Duration_Mins'] = df['Match_Duration_Mins'].fillna(df['Match_Duration_Mins'].median())

# 3. Clean and normalize team and result names (stripping whitespace and standardizing case)
for col in ['Team', 'Result', 'Player']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Print cleaning summary
print("Data cleaning completed.")
print(f"Missing values remaining:\n{df.isnull().sum()}")

Data cleaning completed.
Missing values remaining:
Match_ID               0
Match_Date             0
Team                   0
Opponent               0
Player_Name            0
Player_Points          0
Match_Duration_Mins    0
Match_Type             0
Tournament_Round       0
Result                 0
dtype: int64


In [25]:
import pandas as pd

# 1. Convert Match_Date to datetime
df['Match_Date'] = pd.to_datetime(df['Match_Date'])

# 2. Extract Month (number: 1-12, or name: 'January', 'February', etc.)
df['Month'] = df['Match_Date'].dt.month
df['Month_Name'] = df['Match_Date'].dt.month_name()

# 3. Extract Weekday (number: Monday=0 to Sunday=6, or name: 'Monday', 'Tuesday', etc.)
df['Weekday'] = df['Match_Date'].dt.dayofweek
df['Weekday_Name'] = df['Match_Date'].dt.day_name()


In [28]:
# 1. Total matches played (counts total rows)
total_matches = len(df)

# 2. Total points scored (using the correct column name 'Player_Points')
total_points = df['Player_Points'].sum()

# 3. Average match duration (using the correct column name 'Match_Duration_Mins')
avg_duration = df['Match_Duration_Mins'].mean()

print(f"Total Matches: {total_matches}")
print(f"Total Points: {total_points:.2f}")
print(f"Average Duration: {avg_duration:.2f} mins")

Total Matches: 600
Total Points: 12826.00
Average Duration: 104.97 mins


In [30]:
# 1. Create a helper column for wins (1 if 'Win', else 0)
df['Is_Win'] = (df['Result'] == 'Win').astype(int)

# 2. Group by Team and calculate total points and wins
# Using 'Player_Points' to match the actual dataset schema
team_rankings = df.groupby('Team', as_index=False).agg(
    Total_Points=('Player_Points', 'sum'),
    Matches_Won=('Is_Win', 'sum')
)

# 3. Sort by Total_Points (primary) and Matches_Won (tie-breaker)
team_rankings = team_rankings.sort_values(
    by=['Total_Points', 'Matches_Won'],
    ascending=[False, False]
).reset_index(drop=True)

# 4. Add a 1-based Rank column
team_rankings['Rank'] = team_rankings.index + 1

print(team_rankings)

       Team  Total_Points  Matches_Won  Rank
0    Sharks        2025.0           29     1
1  Panthers        1809.0           28     2
2   Falcons        1754.0           28     3
3  Warriors        1626.0           28     4
4    Titans        1581.0           20     5
5     Lions        1372.0           20     6
6    Eagles        1332.0           25     7
7    Tigers        1327.0           25     8


In [32]:
# Identify the top 10 players based on total points scored
if 'Player_Name' in df.columns and 'Player_Points' in df.columns:
    top_10_players = (
        df.groupby(['Player_Name', 'Team'], as_index=False)
        .agg(
            Total_Points=('Player_Points', 'sum'),
            Matches_Played=('Match_ID', 'nunique')
        )
        .nlargest(10, 'Total_Points')
    )
    print(top_10_players)
else:
    print("Required player columns not found in DataFrame.")

    Player_Name      Team  Total_Points  Matches_Played
277   Player 60  Panthers         145.0               5
377    Player 8    Sharks         137.0               4
62    Player 21   Falcons         128.0               5
318   Player 68    Titans         114.0               5
190   Player 43    Sharks         102.0               3
256   Player 57     Lions         100.0               4
32    Player 16    Titans          94.0               4
202   Player 45  Warriors          93.0               3
88    Player 26    Sharks          92.0               3
335   Player 70    Sharks          90.0               3


In [34]:
# Compare average points, match count, and spread across match results
if 'Result' in df.columns and 'Player_Points' in df.columns:
    result_comparison = (
        df.groupby('Result', as_index=False)
        .agg(
            Total_Matches=('Player_Points', 'count'),
            Avg_Points=('Player_Points', 'mean'),
            Median_Points=('Player_Points', 'median'),
            Std_Dev=('Player_Points', 'std')
        )
        .sort_values(by='Avg_Points', ascending=False)
        .reset_index(drop=True)
    )
    print(result_comparison.round(2))
else:
    print("Required columns ('Result', 'Player_Points') not found in DataFrame.")

  Result  Total_Matches  Avg_Points  Median_Points  Std_Dev
0   Draw            209       22.65           21.0    12.98
1    Win            203       21.03           21.0    12.07
2   Loss            188       20.34           21.0    12.85


In [35]:
# Group by Team to get total matches, wins, and win percentage
# (Note: df['Is_Win'] is already created in a previous cell)
team_win_pct = (
    df.groupby('Team', as_index=False)
    .agg(
        Matches_Played=('Result', 'count'),
        Matches_Won=('Is_Win', 'sum')
    )
)

team_win_pct['Win_Percentage'] = (
    (team_win_pct['Matches_Won'] / team_win_pct['Matches_Played']) * 100
).round(2)

# Sort from highest to lowest win percentage
team_win_pct = team_win_pct.sort_values(
    by=['Win_Percentage', 'Matches_Won'],
    ascending=[False, False]
).reset_index(drop=True)

print(team_win_pct)

       Team  Matches_Played  Matches_Won  Win_Percentage
0  Panthers              72           28           38.89
1    Tigers              65           25           38.46
2  Warriors              75           28           37.33
3    Eagles              69           25           36.23
4    Sharks              90           29           32.22
5   Falcons              90           28           31.11
6     Lions              66           20           30.30
7    Titans              73           20           27.40


In [37]:
# 1. Use the correct column name 'Match_Duration_Mins'
if 'Match_Duration_Mins' in df.columns:
    # 2. Find exact min and max durations
    max_duration = df['Match_Duration_Mins'].max()
    min_duration = df['Match_Duration_Mins'].min()

    # 3. Filter rows matching the longest and shortest durations
    longest_matches = df[df['Match_Duration_Mins'] == max_duration]
    shortest_matches = df[df['Match_Duration_Mins'] == min_duration]

    print("--- Longest Match(es) ---")
    display(longest_matches)

    print("\n--- Shortest Match(es) ---")
    display(shortest_matches)
else:
    print("Column 'Match_Duration_Mins' not found in DataFrame.")

--- Longest Match(es) ---


,Match_ID,Match_Date,Team,Opponent,Player_Name,Player_Points,Match_Duration_Mins,Match_Type,Tournament_Round,Result,Month,Month_Name,Weekday,Weekday_Name,Is_Win
76,MA00076,2025-09-20,Sharks,Sharks,Player 32,7.0,179.0,Semi-Final,Round 1,Draw,9,September,5,Saturday,0
339,MA00339,2026-09-06,Panthers,Falcons,Player 74,26.0,179.0,League,Round 1,Win,9,September,6,Sunday,1
357,MA00357,2026-02-20,Warriors,Falcons,Player 73,35.0,179.0,Semi-Final,Final,Draw,2,February,4,Friday,0
407,MA00407,2025-01-21,Titans,Tigers,Player 39,14.0,179.0,Friendly,Final,Loss,1,January,1,Tuesday,0
521,MA00521,2025-02-15,Sharks,Lions,Player 74,18.0,179.0,Final,Final,Draw,2,February,5,Saturday,0



--- Shortest Match(es) ---


,Match_ID,Match_Date,Team,Opponent,Player_Name,Player_Points,Match_Duration_Mins,Match_Type,Tournament_Round,Result,Month,Month_Name,Weekday,Weekday_Name,Is_Win
17,MA00017,2025-05-21,Lions,Panthers,Player 68,10.0,30.0,Semi-Final,Quarter-Final,Draw,5,May,2,Wednesday,0
19,MA00019,2026-01-21,Panthers,Warriors,Player 56,21.0,30.0,Friendly,Round 2,Draw,1,January,2,Wednesday,0
98,MA00098,2025-06-05,Warriors,Titans,Player 59,0.0,30.0,Friendly,Semi-Final,Loss,6,June,3,Thursday,0
351,MA00351,2026-08-24,Panthers,Panthers,Player 30,30.0,30.0,Semi-Final,Final,Win,8,August,0,Monday,1
549,MA00549,2026-04-05,Warriors,Panthers,Player 68,42.0,30.0,Friendly,Round 1,Win,4,April,6,Sunday,1


In [40]:
# 1. Tuple for fixed, immutable match-result labels
MATCH_RESULTS = ('Win', 'Loss', 'Draw')

# Optional: Validate that the dataset only contains allowed result labels
invalid_results = set(df['Result'].dropna()) - set(MATCH_RESULTS)
if invalid_results:
    print(f"Warning: Unexpected result labels found: {invalid_results}")

# 2. Set to identify all unique participating teams (single 'Team' column)
participating_teams = set(df['Team'].dropna())

print(f"Allowed Results ({type(MATCH_RESULTS).__name__}): {MATCH_RESULTS}")
print(f"Total Unique Teams: {len(participating_teams)}")
print(f"Participating Teams ({type(participating_teams).__name__}): {sorted(participating_teams)}")


Allowed Results (tuple): ('Win', 'Loss', 'Draw')
Total Unique Teams: 8
Participating Teams (set): ['Eagles', 'Falcons', 'Lions', 'Panthers', 'Sharks', 'Tigers', 'Titans', 'Warriors']


In [44]:
# 1. Dynamically retrieve the match-result labels and active participating teams
MATCH_RESULTS = ('Win', 'Loss', 'Draw')
participating_teams = set(df['Team'].dropna())

# 2. Store team-wise wins using the active dataset
win_counts = df[df['Result'] == MATCH_RESULTS[0]]['Team'].value_counts()
team_wins = {team: int(win_counts.get(team, 0)) for team in participating_teams}

# Store total matches played per team dynamically
team_matches = df['Team'].value_counts().to_dict()

# 3. Clean function to calculate a team's win percentage
def get_win_percentage(team_name, wins_dict, matches_dict):
    """Calculate win percentage for a given team dynamically."""
    total_matches = matches_dict.get(team_name, 0)
    if total_matches == 0:
        return 0.0
    wins = wins_dict.get(team_name, 0)
    return round((wins / total_matches) * 100, 2)

# Display cleaned results
print("Team Wins Dictionary:", team_wins)
print("\n--- Team Performance Summary ---")
for team in sorted(participating_teams):
    win_pct = get_win_percentage(team, team_wins, team_matches)
    print(f"{team}: {team_wins[team]} wins / {team_matches[team]} matches ({win_pct}%)")

Team Wins Dictionary: {'Falcons': 28, 'Titans': 20, 'Lions': 20, 'Eagles': 25, 'Tigers': 25, 'Warriors': 28, 'Sharks': 29, 'Panthers': 28}

--- Team Performance Summary ---
Eagles: 25 wins / 69 matches (36.23%)
Falcons: 28 wins / 90 matches (31.11%)
Lions: 20 wins / 66 matches (30.3%)
Panthers: 28 wins / 72 matches (38.89%)
Sharks: 29 wins / 90 matches (32.22%)
Tigers: 25 wins / 65 matches (38.46%)
Titans: 20 wins / 73 matches (27.4%)
Warriors: 28 wins / 75 matches (37.33%)


In [42]:
import numpy as np

# 1. Convert the correct Player_Points column to a 1D NumPy array (dropping missing values if any)
if 'Player_Points' in df.columns:
    points_array = df['Player_Points'].dropna().to_numpy()

    # 2. Calculate mean, median, and standard deviation
    mean_points = np.mean(points_array)
    median_points = np.median(points_array)
    std_points = np.std(points_array)  # Population std (ddof=0) by default

    print(f"Mean Points: {mean_points:.2f}")
    print(f"Median Points: {median_points:.2f}")
    print(f"Standard Deviation: {std_points:.2f}")
else:
    print("Column 'Player_Points' not found in DataFrame.")

Mean Points: 21.38
Median Points: 21.00
Standard Deviation: 12.64


In [46]:
import numpy as np
import pandas as pd

if 'Player_Points' in df.columns and 'Player_Name' in df.columns:
    # 1. Calculate Q1, Q3, IQR, and the upper outlier fence using NumPy
    points = df['Player_Points'].dropna().to_numpy()
    q1 = np.percentile(points, 25)
    q3 = np.percentile(points, 75)
    iqr = q3 - q1

    # Standard Tukey upper fence for unusually high outliers
    upper_fence = q3 + (1.5 * iqr)

    print(f"Q1 (25th percentile): {q1:.2f}")
    print(f"Q3 (75th percentile): {q3:.2f}")
    print(f"IQR: {iqr:.2f}")
    print(f"Unusually High Score Threshold (> Q3 + 1.5*IQR): {upper_fence:.2f}")

    # 2. Identify unusually high player scores
    high_outliers = df[df['Player_Points'] > upper_fence].sort_values(by='Player_Points', ascending=False)
    print("\n--- Unusually High Scores ---")
    display(high_outliers[['Player_Name', 'Player_Points']])

    # 3. Create player-performance groups using documented IQR/quartile thresholds
    bins = [-np.inf, q1, q3, upper_fence, np.inf]
    labels = ['Below Average', 'Average', 'Above Average', 'Elite Outlier']

    df['Performance_Group'] = pd.cut(
        df['Player_Points'],
        bins=bins,
        labels=labels,
        right=True  # Intervals are (a, b]
    )

    print("\n--- Performance Group Distribution ---")
    print(df['Performance_Group'].value_counts())
else:
    print("Required columns ('Player_Points', 'Player_Name') not found in DataFrame.")

Q1 (25th percentile): 11.00
Q3 (75th percentile): 32.00
IQR: 21.00
Unusually High Score Threshold (> Q3 + 1.5*IQR): 63.50

--- Unusually High Scores ---


,Player_Name,Player_Points



--- Performance Group Distribution ---
Performance_Group
Average          289
Below Average    168
Above Average    143
Elite Outlier      0
Name: count, dtype: int64


In [47]:
class Player:
    """Represents a tournament player with basic performance attributes."""

    def __init__(self, name: str, team: str, points: float):
        self.name = name
        self.team = team
        self.points = points

    def add_points(self, new_points: float) -> None:
        """Add points scored in a new match."""
        self.points += new_points

    def __repr__(self) -> str:
        return f"Player(name='{self.name}', team='{self.team}', points={self.points})"


# Example: Creating Player objects from a pandas DataFrame (e.g., top_10_players)
# players = [
#     Player(row['Player'], row['Team'], row['Total_Points'])
#     for _, row in player_summary.iterrows()
# ]

# Standalone usage example
p1 = Player("Aarav Sharma", "Delhi Strikers", 145)
p1.add_points(22)
print(p1)  # Output: Player(name='Aarav Sharma', team='Delhi Strikers', points=167)


Player(name='Aarav Sharma', team='Delhi Strikers', points=167)
